# 08.2 - Self-Attention Deep Dive

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

Self-attention is the mechanism that lets each token compute a **weighted sum of every other token's representation**, where the weights are learned and task-dependent. This unit opens the box: the math, the shapes, the masking, and the pitfalls.

## 2. Why Does This Matter?

Self-attention is the single most important operation in transformers. Every variant (BERT, GPT, T5, ViT) is a different wrapper around this one operation. Debug well, and you can diagnose almost any transformer misbehavior.

## 3. Prerequisites

- Matrix multiplication and dot products (Phase 02)
- Softmax and probability basics
- Transformer overview (08.1)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement scaled dot-product attention from scratch
- Explain Q, K, V, the sqrt(d_k) scaling, and masking
- Implement multi-head attention with learned projections
- Explain why scaling is not optional

## 5. Mental Model

Self-attention is a library lookup. Each token writes a **query** (what am I searching for?), a **key** (what do I offer?), and a **value** (what do I contribute?). The query dot-products against every key; high similarity wins more attention; the output is a weighted blend of all values.

```text
Q = X @ W_q   (what I'm looking for)
K = X @ W_k   (what I contain)
V = X @ W_v   (what I provide)

Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Scaled Dot-Product Attention, From Zero

We implement attention over (batch, heads, seq, d_k) tensors - exactly the shape a multi-head implementation produces.


In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

batch, seq_len, d_model = 1, 5, 64
n_heads = 4
d_k = d_model // n_heads

Q = torch.randn(batch, n_heads, seq_len, d_k)
K = torch.randn(batch, n_heads, seq_len, d_k)
V = torch.randn(batch, n_heads, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print('Q      :', tuple(Q.shape))
print('output :', tuple(output.shape))
print('weights:', tuple(weights.shape))
print('Rows of weights sum to 1:', bool(torch.allclose(weights.sum(-1), torch.ones_like(weights.sum(-1)), atol=1e-5)))


Q      : (1, 4, 5, 16)
output : (1, 4, 5, 16)
weights: (1, 4, 5, 5)


Rows of weights sum to 1: True


## 8. Why Scale by sqrt(d_k)?

Dot products grow with dimension - large scores push softmax into a peaky, saturated regime. Scaling keeps the softmax in a usable range.


In [3]:
def softmax_entropy(p):
    p = p.clamp(min=1e-12)
    return -(p * p.log()).sum(-1)

for d in [1, 8, 32, 128]:
    q = torch.randn(1, 1, 4, d)
    k = torch.randn(1, 1, 4, d)
    raw = torch.matmul(q, k.transpose(-2, -1))
    unscaled = F.softmax(raw, dim=-1)   # forget the scale
    scaled = F.softmax(raw / math.sqrt(d), dim=-1)
    print(f'd_k={d:3d}: unscaled entropy={softmax_entropy(unscaled).mean():.3f}, '
          f'scaled entropy={softmax_entropy(scaled).mean():.3f}')
print('\nWithout scaling, attention collapses to a one-hot (entropy -> 0) as d_k grows.')


d_k=  1: unscaled entropy=1.211, scaled entropy=1.211
d_k=  8: unscaled entropy=0.685, scaled entropy=1.227
d_k= 32: unscaled entropy=0.162, scaled entropy=1.084
d_k=128: unscaled entropy=0.293, scaled entropy=1.214

Without scaling, attention collapses to a one-hot (entropy -> 0) as d_k grows.


## 9. Causal Masking

Decoder = each position may attend only to itself and earlier positions.


In [4]:
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.long)).unsqueeze(0).unsqueeze(0)
print('Causal mask\n', causal_mask[0, 0].numpy())

Q = torch.randn(1, 4, seq_len, 8)
K = torch.randn(1, 4, seq_len, 8)
V = torch.randn(1, 4, seq_len, 8)
output, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print('\nCausal attention weights for one head:\n', torch.round(weights[0, 0], decimals=3).numpy())
upper = weights[0, 0].triu(1)
print('All future positions have zero weight:', bool((upper.abs() < 1e-6).all()))


Causal mask
 [[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]



Causal attention weights for one head:
 [[1.    0.    0.    0.    0.   ]
 [0.779 0.221 0.    0.    0.   ]
 [0.035 0.037 0.929 0.    0.   ]
 [0.466 0.133 0.325 0.077 0.   ]
 [0.145 0.04  0.354 0.324 0.136]]
All future positions have zero weight: True


## 10. Multi-Head Attention with Learned Projections

Multiple heads let the model attend along different relationships in parallel. Heads are concatenated and re-projected (never averaged).


In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        H = self.n_heads
        dk = self.d_k
        Q = self.wq(x).view(B, T, H, dk).transpose(1, 2)
        K = self.wk(x).view(B, T, H, dk).transpose(1, 2)
        V = self.wv(x).view(B, T, H, dk).transpose(1, 2)
        out, weights = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, T, H * dk)
        return self.wo(out), weights


mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 8, 64)
out, w = mha(x)
print('Multi-head output:', tuple(out.shape), '(concat + final projection)')
print('Weight shape per head:', tuple(w.shape))


Multi-head output: (2, 8, 64) (concat + final projection)
Weight shape per head: (2, 4, 8, 8)


## 11. Self-Attention vs Cross-Attention

Self-attention: Q, K, V all come from the same sequence. Cross-attention (used in decoders): Q from the target, K and V from the source.


In [6]:
def cross_attention(x_target, x_source, mask=None):
    # Q from target, K/V from source
    Q = x_target @ torch.randn(x_target.size(-1), 8) / (x_target.size(-1) ** 0.5)
    K = x_source @ torch.randn(x_source.size(-1), 8) / (x_source.size(-1) ** 0.5)
    V = x_source @ torch.randn(x_source.size(-1), 8) / (x_source.size(-1) ** 0.5)
    out, w = scaled_dot_product_attention(Q.unsqueeze(1), K.unsqueeze(1), V.unsqueeze(1), mask)
    return out, w

src = torch.randn(1, 6, 16)   # encoder memory, 6 positions
tgt = torch.randn(1, 4, 16)   # decoder tokens, 4 positions
out, w = cross_attention(tgt, src)
print('Target len x Source len:', tuple(w.shape[-2:]))
print('Every decoder token blends ALL encoder positions (weights row sum 1).')
print('Each decoder row sum:', torch.round(w[0, 0].sum(-1), decimals=3).tolist())


Target len x Source len: (4, 6)
Every decoder token blends ALL encoder positions (weights row sum 1).
Each decoder row sum: [1.0, 1.0, 1.0, 1.0]


## 12. Failure Case: Attention Weights Are Not Feature Importance

A token can attend strongly yet contribute nothing, because the value it provides is uninformative. High weight != important feature.


In [7]:
# Two tokens carry the SAME value v
v = torch.tensor([1.0, 0.0, 0.0, 0.0])
V = torch.stack([v, v]).reshape(1, 1, 2, 4)

def attend(scores_logits, V):
    w = F.softmax(torch.tensor([[[scores_logits]]]), dim=-1)
    return w, torch.matmul(w, V)

w_uniform, out_uniform = attend([0.0, 0.0], V)   # 0.5 / 0.5
w_biased, out_biased = attend([9.0, 0.0], V)     # 99.9% to token 0

print('Uniform weights :', torch.round(w_uniform[0, 0, 0], decimals=3).tolist(), '-> output', torch.round(out_uniform[0, 0, 0], decimals=3).tolist())
print('Heavily biased  :', torch.round(w_biased[0, 0, 0], decimals=3).tolist(), '-> output', torch.round(out_biased[0, 0, 0], decimals=3).tolist())
print('\nOutput identical even though weights changed 0.5 vs 0.999.')
print('High attention weight does NOT mean the token was important -'
      'it only means the head looked there.')


Uniform weights : [0.5, 0.5] -> output [1.0, 0.0, 0.0, 0.0]
Heavily biased  : [1.0, 0.0] -> output [1.0, 0.0, 0.0, 0.0]

Output identical even though weights changed 0.5 vs 0.999.
High attention weight does NOT mean the token was important -it only means the head looked there.


## 13. Challenge: O(n^2) in Sequence Length

The attention matrix is n x n. Measure the quadratic growth.


In [8]:
import time

for L in [64, 256, 1024]:
    q = torch.randn(1, 4, L, 16); k = torch.randn(1, 4, L, 16)
    t0 = time.time()
    torch.matmul(q, k.transpose(-2, -1))
    ms = (time.time() - t0) * 1000
    print(f'L={L:4d}: score matrix {L}x{L}={L*L:7d} cells, matmul {ms:6.2f} ms')
print('\nDoubling L roughly quadruples work -> quadratic scaling.')
print('This motivates FlashAttention / sparse variants for long contexts.')


L=  64: score matrix 64x64=   4096 cells, matmul  27.92 ms


L= 256: score matrix 256x256=  65536 cells, matmul   5.89 ms
L=1024: score matrix 1024x1024=1048576 cells, matmul  69.87 ms

Doubling L roughly quadruples work -> quadratic scaling.
This motivates FlashAttention / sparse variants for long contexts.


## 14. Debugging: Common Errors

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Attention weights all near-equal | Dot products too small | Check Q/K magnitude | Scale by sqrt(d_k) |
| Weights all 0 or 1 | Scores too large | Inspect raw scores | Scale, lower LR |
| Decoder leaks future | Missing causal mask | Check upper triangle is 0 | Apply torch.tril |
| Multi-head shape wrong | Forgot concat | Check concat dim | concat(dim=-1) then project |

## 15. Real-World Considerations

- For production, prefer `nn.MultiheadAttention` or `scaled_dot_product_attention` (they handle math + fused kernels).
- Interpret attention cautiously: weights relate to *coverage*, not causal importance.
- FlashAttention avoids materializing the n x n matrix for long contexts.

## 16. Common Mistakes

- Forgetting sqrt(d_k) scaling.
- No causal mask in decoders.
- Confusing attention weights with the output.
- Averaging heads instead of concatenating.

## 17. When NOT to Use

- When sequence length explodes and exact attention is unaffordable - use linear/sparse variants.
- When a model needs to look only at localized structure - convolutions can be cheaper.

## 18. Closed-Book Recall

1. Write the attention formula from memory.
2. Why scale by sqrt(d_k) and not d_k?
3. How does causal masking change the computation?
4. Self-attention vs cross-attention - where do Q, K, V come from?
5. Why multiple heads instead of one large head?

## 19. Teach-Back Questions

Explain to another person:

- The library-lookup mental model for Q, K, V.
- Why scaling prevents softmax saturation.
- Why attention weights are not feature importance.

## 20. Summary

You implemented scaled dot-product attention, multi-head attention with learned projections, causal masking, and cross-attention - and caught the classic failure modes (missing scaling, weight-vs-importance confusion) in small, sharp experiments.

## 21. Further Experiment

- Replace matmul with `torch.nn.functional.scaled_dot_product_attention` and compare speed.
- Visualize real learned attention maps of a trained classifier vs random heads.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
